# 00 — Dataset Overview

**File:** `go_filtration_data.csv`  
**Coverage:** 55 articles, 573 rows (after export from master Excel database)  

The dataset aggregates GO membrane nanofiltration data from the literature:
- **Rejection** rows: salt (NaCl, MgSO₄, …) and dye (MB, CR, …) rejection experiments
- **Permeance** rows: water, salt, and dye permeance experiments

Each row is one filtration experiment. Multiple rows per article are common (different pressures, concentrations, or solutes on the same membrane).

**Working directory:** run from `scripts_for_github/` — data file is one level up (`../go_filtration_data.csv`).

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

DATA_FILE = os.path.join('..', 'go_filtration_data.csv')
df = pd.read_csv(DATA_FILE, encoding='utf-8-sig')

# coerce numeric columns
NUM_COLS = [
    'pressure_bar', 'solution_concentration_M', 'Rejection_%', 'Permeance_L_m2_bar_h',
    'MW', 'formal_charge', 'cation_charge', 'anion_charge',
    'cation_hydrated_radius_nm', 'anion_hydrated_radius_nm',
    'thickness_nm', 'flake_size_nm', 'dry_dspacing_nm', 'wet_dspacing_nm',
    'C-C', 'C-O', 'C=O', 'C:O_from_C1s',
    'Zeta_potential_mV', 'Water_contact_angle',
]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

print(f'Total rows: {len(df)}')
print(f'Articles (DOIs): {df["doi"].nunique()}')
print(f'Unique membranes: {df["membrane_uid"].nunique()}')
print(f'\nRow counts by type:')
print(df['type'].value_counts().to_string())

## Subsets used in modelling

After `dropna` on required features and deduplication by `(doi + features + target)`:

| Task | Subset | n | DOIs |
|------|--------|---|------|
| Rejection — salts | type == 'salt', feature set C | 270 | ~40 |
| Rejection — dyes | type == 'dye', feature set C | 111 | ~27 |
| Permeance | type ∈ {water, salt, dye}, struct. feats | 250 | ~38 |
| GO properties (Section 3.2) | unique membranes | varies | — |

In [ ]:
# --- Dataset size after deduplication for each major analysis ---
FEATS_SALT = ['pressure_bar', 'solution_concentration_M',
              'cation_charge', 'anion_charge',
              'cation_hydrated_radius_nm', 'anion_hydrated_radius_nm',
              'thickness_nm', 'flake_size_nm', 'dry_dspacing_nm',
              'C-C', 'C-O', 'C=O']
FEATS_DYE  = ['pressure_bar', 'solution_concentration_M',
              'MW', 'formal_charge',
              'thickness_nm', 'flake_size_nm', 'dry_dspacing_nm',
              'C-C', 'C-O', 'C=O']
FEATS_PERM = ['pressure_bar', 'thickness_nm', 'flake_size_nm',
              'dry_dspacing_nm', 'C-C', 'C-O', 'C=O']

def subset_size(df, type_filter, feats, target):
    sub = df[df['type'].isin(type_filter)].copy()
    sub = sub.dropna(subset=feats + [target])
    sub = sub.drop_duplicates(subset=['doi'] + feats + [target])
    return len(sub), sub['doi'].nunique()

n_s, d_s = subset_size(df, ['salt'],               FEATS_SALT, 'Rejection_%')
n_d, d_d = subset_size(df, ['dye'],                FEATS_DYE,  'Rejection_%')
n_p, d_p = subset_size(df, ['water','salt','dye'], FEATS_PERM, 'Permeance_L_m2_bar_h')

print(f'Salt rejection  (feature set C): n={n_s:3d}, {d_s} DOIs')
print(f'Dye  rejection  (feature set C): n={n_d:3d}, {d_d} DOIs')
print(f'Permeance       (struct. feats): n={n_p:3d}, {d_p} DOIs')

## Feature groups

| Group | Features | Notes |
|-------|----------|-------|
| Filtration | pressure_bar, concentration_M | Operating conditions |
| Ionic solute | cation/anion charge, cation/anion radius | Salts only |
| Molecular solute | MW, formal_charge | Dyes only |
| Structure | thickness_nm, flake_size_nm, dry_dspacing_nm | Membrane geometry |
| XPS (C1s) | C–C, C–O, C=O | Bond fractions; C:O ratio **excluded** (collinear) |
| Surface | Zeta_potential_mV, Water_contact_angle | Feature set D only |
| Wet d-spacing | wet_dspacing_nm | Measured or Ridge-imputed (hybrid) |

In [ ]:
# --- Missing value fraction for each feature ---
features_to_check = {
    'pressure_bar':              'Operating',
    'solution_concentration_M':  'Operating',
    'Rejection_%':               'Target',
    'Permeance_L_m2_bar_h':      'Target',
    'MW':                        'Dye solute',
    'formal_charge':             'Dye solute',
    'cation_charge':             'Salt solute',
    'anion_charge':              'Salt solute',
    'cation_hydrated_radius_nm': 'Salt solute',
    'anion_hydrated_radius_nm':  'Salt solute',
    'thickness_nm':              'Structure',
    'flake_size_nm':             'Structure',
    'dry_dspacing_nm':           'Structure',
    'wet_dspacing_nm':           'Structure',
    'C-C':                       'XPS',
    'C-O':                       'XPS',
    'C=O':                       'XPS',
    'Zeta_potential_mV':         'Surface',
    'Water_contact_angle':       'Surface',
}

rows = []
for feat, grp in features_to_check.items():
    if feat in df.columns:
        n_avail = df[feat].notna().sum()
        rows.append({'Feature': feat, 'Group': grp,
                     'Missing': len(df)-n_avail,
                     'Missing (%)': round(100 * (1 - n_avail / len(df)), 1)})

miss_df = pd.DataFrame(rows)
print(miss_df.to_string(index=False))

In [ ]:
# --- Articles per supplier type ---
if 'supplier' in df.columns:
    df['source'] = df['supplier'].str.lower().str.startswith('self-synth').map(
        {True: 'Self-synth', False: 'Commercial'})
    mem_df = df.drop_duplicates(subset=['doi', 'membrane_uid'])
    print('Unique membranes by GO source:')
    print(mem_df['source'].value_counts().to_string())
    print(f'\nTotal unique membranes: {len(mem_df)}')